In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchaudio
from torchvision.models import resnet18
from torch.utils.data import DataLoader
from tqdm import tqdm
import pickle

from collections import Counter

In [2]:
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 0.001
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
DATA_DIR = "./data"
print("Device: ", DEVICE)

Device:  cuda:3


In [3]:
train_dataset = torchaudio.datasets.SPEECHCOMMANDS(DATA_DIR, url='speech_commands_v0.02', subset='training', download=True)
test_dataset = torchaudio.datasets.SPEECHCOMMANDS(DATA_DIR, url='speech_commands_v0.02', subset='testing', download=True)

In [4]:
with open("/home/piskunovvs/effml/project/labels.pickle", "rb") as file:
    labels = pickle.load(file)

In [5]:
label_to_index = {label: index for index, label in enumerate(set(labels))}
num_classes = len(set(labels))
print(f"Found {num_classes} classes: {set(labels)}")

Found 35 classes: {'wow', 'go', 'learn', 'one', 'cat', 'forward', 'zero', 'stop', 'happy', 'four', 'follow', 'down', 'dog', 'nine', 'yes', 'left', 'marvin', 'bed', 'tree', 'off', 'backward', 'on', 'right', 'house', 'six', 'three', 'two', 'up', 'eight', 'bird', 'sheila', 'five', 'visual', 'seven', 'no'}


In [59]:
# import pickle
# with open("labels.pickle", "wb") as f:
#     pickle.dump(labels, f)

In [6]:
def collate_fn(batch):
    tensors, targets = [], []
    for waveform, _, label, _, _ in batch:
        if waveform.shape[1] < SAMPLE_RATE:
            padding = torch.randn(1, SAMPLE_RATE - waveform.shape[1]) * 1e-6
            waveform = torch.cat([waveform, padding], dim=1)
        elif waveform.shape[1] > SAMPLE_RATE:
            waveform = waveform[:, :SAMPLE_RATE]
            
        tensors.append(waveform)
        targets.append(label_to_index[label])

    tensors = torch.stack(tensors)
    targets = torch.tensor(targets, dtype=torch.long)
    return tensors, targets

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)

In [7]:
transform = nn.Sequential(
    torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=64),
    torchaudio.transforms.AmplitudeToDB()
).to(DEVICE)

In [8]:
model = resnet18(weights=None)

model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

In [9]:
def train(model, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for waveforms, labels in pbar:
        waveforms, labels = waveforms.to(DEVICE), labels.to(DEVICE)
        
        spectrograms = transform(waveforms)
        
        optimizer.zero_grad()
        outputs = model(spectrograms)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'Loss': running_loss/total, 'Acc': 100.*correct/total})

def test(model, epoch):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        pbar = tqdm(test_loader, desc=f"Epoch {epoch}/{EPOCHS} [Test ]")
        for waveforms, labels in pbar:
            waveforms, labels = waveforms.to(DEVICE), labels.to(DEVICE)
            
            spectrograms = transform(waveforms)
            outputs = model(spectrograms)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({'Loss': running_loss/total, 'Acc': 100.*correct/total})
            
    print(f"--> Test Accuracy after Epoch {epoch}: {100.*correct/total:.2f}%\n")

In [10]:
for epoch in range(1, EPOCHS + 1):
    train(model, epoch)
    test(model, epoch)
    
# Save the trained model
torch.save(model.state_dict(), "resnet18_speech_commands_full_precision.pt")

Epoch 1/10 [Test ]: 100%|██████████| 86/86 [00:08<00:00,  9.69it/s, Loss=0.00355, Acc=85.8]


--> Test Accuracy after Epoch 1: 85.81%



Epoch 2/10 [Test ]: 100%|██████████| 86/86 [00:08<00:00, 10.22it/s, Loss=0.00207, Acc=92.1]


--> Test Accuracy after Epoch 2: 92.09%



Epoch 3/10 [Test ]: 100%|██████████| 86/86 [00:08<00:00, 10.36it/s, Loss=0.00229, Acc=91.1]


...

--> Test Accuracy after Epoch 8: 94.54%



Epoch 9/10 [Test ]: 100%|██████████| 86/86 [00:07<00:00, 10.87it/s, Loss=0.00155, Acc=94.3]


--> Test Accuracy after Epoch 9: 94.32%



Epoch 10/10 [Test ]: 100%|██████████| 86/86 [00:08<00:00, 10.66it/s, Loss=0.00132, Acc=94.8]


--> Test Accuracy after Epoch 10: 94.80%



In [11]:
# teacher_model = resnet18(weights=None)
# teacher_model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
# teacher_model.fc = nn.Linear(model.fc.in_features, num_classes)

# teacher_model.load_state_dict(torch.load("resnet18_speech_commands_full_precision.pt", weights_only=True))
# teacher_model = teacher_model.to(DEVICE)
# teacher_model.eval()

teacher_model = model
teacher_model.eval()

for param in teacher_model.parameters():
    param.requires_grad = False

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

class BinarizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)

        return torch.where(input >= 0, 
                           torch.tensor(1.0, device=input.device), 
                           torch.tensor(-1.0, device=input.device))

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad_input[input.abs() > 1.0] = 0 
        return grad_input

def binarize(x):
    return BinarizeSTE.apply(x)

In [13]:
class BinaryConv2d(nn.Conv2d):
    def forward(self, input):
        binarized_weight = binarize(self.weight)
        return F.conv2d(input, binarized_weight, self.bias, self.stride, 
                        self.padding, self.dilation, self.groups)

class BinaryBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BinaryBasicBlock, self).__init__()
        self.conv1 = BinaryConv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        
        self.conv2 = BinaryConv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                BinaryConv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = binarize(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(x)
        out = binarize(out)
        return out


class BinaryResNet18(nn.Module):
    def __init__(self, num_classes=35):
        super(BinaryResNet18, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.fc = nn.Linear(512 * BinaryBasicBlock.expansion, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(BinaryBasicBlock(self.in_planes, planes, s))
            self.in_planes = planes * BinaryBasicBlock.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.maxpool(F.relu(self.bn1(self.conv1(x))))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

In [14]:
def distillation_loss(student_logits, teacher_logits, labels, T=2.0, alpha=0.5):
    soft_targets = F.softmax(teacher_logits / T, dim=-1)
    student_log_probs = F.log_softmax(student_logits, dim=-1)
    distil_loss = F.kl_div(student_log_probs, soft_targets, reduction='batchmean') * (T * T)
    
    hard_loss = F.cross_entropy(student_logits, labels)
    
    return alpha * hard_loss + (1.0 - alpha) * distil_loss


In [15]:
def train_bnn_with_distillation(student_model, teacher_model, train_loader, optimizer, transform, device, epoch):
    student_model.train()
    teacher_model.eval()
    
    running_loss = 0.
    total = 0
    correct = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for batch_idx, (waveforms, labels) in enumerate(pbar):
        waveforms, labels = waveforms.to(device), labels.to(device)
        
        specs = transform(waveforms)
        
        with torch.no_grad():
            teacher_logits = teacher_model(specs)

        optimizer.zero_grad()
        student_logits = student_model(specs)

        loss = distillation_loss(student_logits, teacher_logits, labels, T=3.0, alpha=0.3)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        pred = student_logits.argmax(dim=1, keepdim=True)

        correct += pred.eq(labels.view_as(pred)).sum().item()

        total += labels.size(0)
            
        pbar.set_postfix({'Loss': running_loss/total, 'Acc': 100.*correct/total})



In [16]:
from tqdm import trange

# TODO add scaling to modules

In [17]:
student_model = BinaryResNet18(num_classes=35).to(DEVICE)
optimizer = torch.optim.AdamW(student_model.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(EPOCHS):
    train_bnn_with_distillation(student_model, teacher_model, train_loader, optimizer, transform, DEVICE, epoch)
    test(student_model, epoch)

Epoch 0/10 [Test ]: 100%|██████████| 86/86 [00:08<00:00, 10.40it/s, Loss=0.0066, Acc=81.4] 


--> Test Accuracy after Epoch 0: 81.43%



Epoch 1/10 [Test ]: 100%|██████████| 86/86 [00:08<00:00, 10.21it/s, Loss=0.00552, Acc=86]  


--> Test Accuracy after Epoch 1: 86.02%



Epoch 2/10 [Test ]: 100%|██████████| 86/86 [00:08<00:00, 10.25it/s, Loss=0.0049, Acc=88]   


...

--> Test Accuracy after Epoch 7: 91.49%



Epoch 8/10 [Test ]: 100%|██████████| 86/86 [00:08<00:00, 10.52it/s, Loss=0.00427, Acc=91.8]


--> Test Accuracy after Epoch 8: 91.85%



Epoch 9/10 [Test ]: 100%|██████████| 86/86 [00:08<00:00, 10.58it/s, Loss=0.00416, Acc=91.8]

--> Test Accuracy after Epoch 9: 91.80%



In [18]:
import torch
import numpy as np

def save_1bit_model(model, filepath):
    state_dict = model.state_dict()
    packed_dict = {}

    binary_layers = [name for name, mod in model.named_modules() if isinstance(mod, BinaryConv2d)]

    for key, tensor in state_dict.items():
        is_binary_weight = any(key == f"{b_name}.weight" for b_name in binary_layers)

        if is_binary_weight:
            w = tensor.detach()
            
            # Calculate the XNOR scale factor
            scale = w.abs().mean(dim=(1, 2, 3), keepdim=True)
            
            # Binarize to bool
            w_bool = (w >= 0).cpu().numpy().astype(np.uint8)
            
            # Pack 8 booleans into a single byte
            w_packed = np.packbits(w_bool.reshape(-1))

            # Save the compressed bits, the scale, and the original shape
            packed_dict[key + '_packed'] = w_packed
            packed_dict[key + '_scale'] = scale.cpu()
            packed_dict[key + '_shape'] = w.shape
        else:
            packed_dict[key] = tensor.cpu()

    torch.save(packed_dict, filepath)
    print(f"✅ 1-Bit Model saved to {filepath}")


In [20]:
sum(p.numel() for p in teacher_model.parameters()), sum(p.numel() for p in student_model.parameters())

(11188195, 11188195)

In [21]:
save_1bit_model(student_model, "resnet18_1bit.pt")

✅ 1-Bit Model saved to resnet18_1bit.pt


In [24]:
import os

print(os.path.getsize("resnet18_speech_commands_full_precision.pt") / (1024 * 1024))
print(os.path.getsize("resnet18_1bit.pt") / (1024 * 1024))

42.75910472869873
2.2062482833862305
